# 🏆 VIETNAMESE LEGAL INFORMATION RETRIEVAL PIPELINE
## DSC Champions - Automated Two-Stage Retrieval with Deep Re-ranking

---
### 📌 Pipeline Overview
1. **Preprocessing & Word Segmentation**: Unicode NFC Normalization, Vietnamese Word Segmentation (`PyVi`/`Underthesea`), Sliding Window Chunking.
2. **First-Stage Hybrid Retrieval**: Lexical (`BM25Okapi`) + Semantic Dense Vector (`vietnamese-bi-encoder`).
3. **Score Normalization & Fusion**: Min-Max scaling and weighted score combination.
4. **Second-Stage Deep Re-ranking**: Cross-Encoder (`bge-reranker-v2-m3`) with GPU auto-detection and top-K candidate pruning for ultra-fast inference.
5. **Post-Processing & Dynamic Thresholding**: Adaptive threshold selection ensuring strictly $1 \le \text{len}(\text{answer}) \le 5$.
6. **Validation & Submission Packaging**: Evaluates Mean Recall / Precision and creates `submission.zip` containing `submission.json`.

In [ ]:
# =====================================================================
# 1. SYSTEM IMPORTS & HARDWARE ACCELERATION SETUP
# =====================================================================
import os
import sys
import glob
import json
import pickle
import random
import re
import unicodedata
import zipfile
from pathlib import Path
from typing import List, Dict, Tuple, Set, Union, Any

import numpy as np
import torch
from tqdm.auto import tqdm

# Device Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[System] Execution Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"[System] GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print(f"[System] Running on CPU. Optimized for multi-threaded inference.")
    torch.set_num_threads(os.cpu_count() or 4)

# Check Optional Libraries
try:
    from pyvi import ViTokenizer
    HAS_PYVI = True
except ImportError:
    HAS_PYVI = False

try:
    from rank_bm25 import BM25Okapi
    HAS_RANK_BM25 = True
except ImportError:
    HAS_RANK_BM25 = False

try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    HAS_SENTENCE_TRANSFORMERS = True
    HAS_CROSS_ENCODER = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False
    HAS_CROSS_ENCODER = False

print(f"[Dependencies] PyVi: {HAS_PYVI} | Rank-BM25: {HAS_RANK_BM25} | SentenceTransformers: {HAS_SENTENCE_TRANSFORMERS} | CrossEncoder: {HAS_CROSS_ENCODER}")

In [ ]:
# =====================================================================
# 2. GLOBAL CONFIGURATION & AUTO PATH DETECTION (LOCAL & KAGGLE)
# =====================================================================
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    print("[Environment] Running on Kaggle Notebook")
    K_INPUT = Path("/kaggle/input")
    
    # Auto-detect train.json
    train_files = list(K_INPUT.glob("**/train.json")) + list(K_INPUT.glob("**/train*.json"))
    TRAIN_PATH = train_files[0] if train_files else K_INPUT / "train-data" / "train.json"
    
    # Auto-detect public-official.json
    test_files = list(K_INPUT.glob("**/public-official.json")) + list(K_INPUT.glob("**/public*.json")) + list(K_INPUT.glob("**/test*.json"))
    PUBLIC_TEST_PATH = test_files[0] if test_files else K_INPUT / "public-official" / "public-official.json"
    
    # Auto-detect context JSON files folder
    ctx_files = list(K_INPUT.glob("**/context_*.json"))
    CORPUS_DIR = ctx_files[0].parent if ctx_files else K_INPUT / "selected-contexts"
    
    CACHE_DIR = Path("/kaggle/working/cache")
    SUBMISSION_PATH = Path("/kaggle/working/submission.json")
    SUBMISSION_ZIP_PATH = Path("/kaggle/working/submission.zip")
else:
    print("[Environment] Running on Local Environment")
    BASE_DIR = Path(".").resolve()
    DATA_DIR = BASE_DIR
    CORPUS_DIR = DATA_DIR / "selected-contexts" / "selected-contexts"
    TRAIN_PATH = DATA_DIR / "train.json"
    PUBLIC_TEST_PATH = DATA_DIR / "public-official.json"
    CACHE_DIR = BASE_DIR / "cache"
    SUBMISSION_PATH = BASE_DIR / "submission.json"
    SUBMISSION_ZIP_PATH = BASE_DIR / "submission.zip"

CACHE_DIR.mkdir(exist_ok=True, parents=True)
PROCESSED_CORPUS_PATH = CACHE_DIR / "processed_corpus.json"
BM25_INDEX_PATH = CACHE_DIR / "bm25_index.pkl"
DENSE_INDEX_PATH = CACHE_DIR / "dense_embeddings.npy"

print(f"[Paths] Train Data   : {TRAIN_PATH}")
print(f"[Paths] Public Test  : {PUBLIC_TEST_PATH}")
print(f"[Paths] Contexts Dir : {CORPUS_DIR}")
print(f"[Paths] Cache Output : {CACHE_DIR}")

# Preprocessing Parameters
MAX_CHUNK_TOKENS = 350
CHUNK_OVERLAP = 50

# First-Stage Retrieval (BM25 + Dense)
BM25_K1 = 1.5
BM25_B = 0.75
FIRST_STAGE_TOP_K = 30

# Models Configuration (Pretrained: ~703M parameters total << 4B limit)
DENSE_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"  # ~135M params
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"                   # ~568M params

# Speedup Optimization Parameters
RERANK_TOP_K = 15                 # Only rerank Top 15 candidates (10x speedup)
RERANK_CHUNKS_PER_DOC = 1         # Best chunk per candidate
RERANK_MAX_LENGTH = 256           # Compact token length for fast Cross-Encoder inference

# Post-Processing Parameters
DYNAMIC_THRESHOLD_RATIO = 0.88    # Accept docs with score >= top1_score * 0.88
MAX_PREDICTED_DOCS = 3            # Target max documents to predict

# Validation
VAL_RATIO = 0.2
RANDOM_SEED = 42

In [ ]:
# =====================================================================
# 3. UTILITY FUNCTIONS & EVALUATION METRICS
# =====================================================================
def load_json(file_path: Union[str, Path]) -> dict:
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data: dict, file_path: Union[str, Path], indent: int = 4) -> None:
    path = Path(file_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=indent)

def normalize_vietnamese_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[\t\f\v]', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    text = ' '.join(lines)
    return re.sub(r'\s+', ' ', text).strip()

def word_segment(text: str) -> str:
    text = normalize_vietnamese_text(text)
    if HAS_PYVI:
        return ViTokenizer.tokenize(text)
    return text

def compute_sample_recall(y_true: Set[str], y_pred: Set[str]) -> float:
    if not y_true or len(y_pred) > 5:
        return 0.0
    return len(y_true.intersection(y_pred)) / len(y_true)

def compute_sample_precision(y_true: Set[str], y_pred: Set[str]) -> float:
    if not y_pred or len(y_pred) > 5:
        return 0.0
    return len(y_true.intersection(y_pred)) / len(y_pred)

def evaluate_predictions(ground_truth: Dict[str, Any], predictions: Dict[str, Any]) -> Tuple[float, float]:
    total_recall = 0.0
    total_precision = 0.0
    count = 0
    for q_id, sample in ground_truth.items():
        true_answers = set(str(ans) for ans in sample.get('answer', []))
        if not true_answers:
            continue
        pred_sample = predictions.get(q_id, {})
        pred_answers = set(str(ans) for ans in pred_sample.get('answer', []))
        total_recall += compute_sample_recall(true_answers, pred_answers)
        total_precision += compute_sample_precision(true_answers, pred_answers)
        count += 1
    if count == 0:
        return 0.0, 0.0
    return total_recall / count, total_precision / count

In [ ]:
# =====================================================================
# 4. STEP 1: PREPROCESSING & SLIDING WINDOW CHUNKING
# =====================================================================
def chunk_passage(text: str, max_words: int = MAX_CHUNK_TOKENS, overlap: int = CHUNK_OVERLAP) -> List[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += (max_words - overlap)
    return chunks

def process_corpus(force_reprocess: bool = False) -> List[Dict[str, Any]]:
    if not force_reprocess and os.path.exists(PROCESSED_CORPUS_PATH):
        try:
            cached_data = load_json(PROCESSED_CORPUS_PATH)
            if cached_data and len(cached_data) > 0:
                print(f"[Preprocessing] Loaded cached corpus ({len(cached_data)} chunks).")
                return cached_data
        except Exception:
            pass

    file_list = glob.glob(str(CORPUS_DIR / "context_*.json"))
    if not file_list:
        file_list = glob.glob(str(CORPUS_DIR / "**" / "context_*.json"), recursive=True)
    if not file_list and IS_KAGGLE:
        file_list = glob.glob("/kaggle/input/**/context_*.json", recursive=True)
    
    print(f"[Preprocessing] Processing {len(file_list)} legal documents...")
    processed_chunks = []
    
    for file_path in tqdm(file_list, desc="Chunking Documents"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                doc_data = json.load(f)
            doc_id = str(doc_data.get('id', ''))
            doc_name = doc_data.get('name', '')
            passage = doc_data.get('passage', '')
            if not doc_id or not passage:
                continue
            cleaned = normalize_vietnamese_text(passage)
            chunks = chunk_passage(cleaned, max_words=MAX_CHUNK_TOKENS, overlap=CHUNK_OVERLAP)
            for i, c_text in enumerate(chunks):
                combined = f"{doc_name}. {c_text}" if doc_name else c_text
                processed_chunks.append({
                    "chunk_id": f"{doc_id}_{i}",
                    "doc_id": doc_id,
                    "doc_name": doc_name,
                    "text": combined,
                    "segmented_text": word_segment(combined)
                })
        except Exception:
            continue

    print(f"[Preprocessing] Created {len(processed_chunks)} chunks.")
    if processed_chunks:
        save_json(processed_chunks, PROCESSED_CORPUS_PATH, indent=2)
    return processed_chunks

In [ ]:
# =====================================================================
# 5. STEP 2: LEXICAL RETRIEVAL (BM25OKAPI)
# =====================================================================
class SimpleBM25:
    """Fallback BM25 implementation if rank_bm25 is unavailable."""
    def __init__(self, corpus: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.avgdl = sum(len(d) for d in corpus) / self.corpus_size if self.corpus_size > 0 else 1.0
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []
        df = {}
        for doc in corpus:
            self.doc_len.append(len(doc))
            freqs = {}
            for w in doc:
                freqs[w] = freqs.get(w, 0) + 1
            self.doc_freqs.append(freqs)
            for w in set(doc):
                df[w] = df.get(w, 0) + 1
        for w, freq in df.items():
            self.idf[w] = np.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1)

    def get_scores(self, query: List[str]) -> np.ndarray:
        scores = np.zeros(self.corpus_size)
        for q in query:
            if q not in self.idf:
                continue
            q_idf = self.idf[q]
            for idx, doc_freq in enumerate(self.doc_freqs):
                if q in doc_freq:
                    freq = doc_freq[q]
                    num = freq * (self.k1 + 1)
                    den = freq + self.k1 * (1 - self.b + self.b * (self.doc_len[idx] / self.avgdl))
                    scores[idx] += q_idf * (num / den)
        return scores

class BM25Retriever:
    def __init__(self, k1: float = BM25_K1, b: float = BM25_B):
        self.k1 = k1
        self.b = b
        self.bm25 = None
        self.doc_ids = []

    def build_index(self, corpus_chunks: List[Dict[str, Any]], force_rebuild: bool = False):
        self.doc_ids = [c["doc_id"] for c in corpus_chunks]
        if not force_rebuild and os.path.exists(BM25_INDEX_PATH):
            try:
                with open(BM25_INDEX_PATH, 'rb') as f:
                    saved = pickle.load(f)
                    bm25_obj = saved.get("bm25")
                    if bm25_obj is not None and (len(corpus_chunks) == 0 or getattr(bm25_obj, "corpus_size", len(corpus_chunks)) == len(corpus_chunks)):
                        print(f"[BM25] Loaded cached index.")
                        self.bm25 = bm25_obj
                        return
            except Exception:
                pass

        if not corpus_chunks:
            return
        print(f"[BM25] Building BM25 index over {len(corpus_chunks)} chunks...")
        tokenized = [c["segmented_text"].lower().split() for c in corpus_chunks]
        if HAS_RANK_BM25:
            self.bm25 = BM25Okapi(tokenized, k1=self.k1, b=self.b)
        else:
            self.bm25 = SimpleBM25(tokenized, k1=self.k1, b=self.b)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({"bm25": self.bm25}, f)

    def search(self, question: str, top_k: int = FIRST_STAGE_TOP_K) -> List[Tuple[str, float]]:
        if self.bm25 is None or len(self.doc_ids) == 0:
            return []
        scores = self.bm25.get_scores(word_segment(question).lower().split())
        doc_scores: Dict[str, float] = {}
        for idx, score in enumerate(scores):
            if idx < len(self.doc_ids):
                d_id = self.doc_ids[idx]
                if d_id not in doc_scores or score > doc_scores[d_id]:
                    doc_scores[d_id] = float(score)
        return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [ ]:
# =====================================================================
# 6. STEP 3: DENSE VECTOR RETRIEVAL (BI-ENCODER)
# =====================================================================
class DenseRetriever:
    def __init__(self, model_name: str = DENSE_MODEL_NAME):
        self.model_name = model_name
        self.model = None
        self.corpus_embeddings = None
        self.doc_ids = []
        self.device = DEVICE

    def load_model(self):
        if self.model is None and HAS_SENTENCE_TRANSFORMERS:
            try:
                print(f"[Dense] Loading Bi-Encoder model ({self.device}): {self.model_name}")
                self.model = SentenceTransformer(self.model_name, device=self.device)
            except Exception as e:
                print(f"[Dense] Warning: Could not load model '{self.model_name}': {e}")
                self.model = None

    def build_index(self, corpus_chunks: List[Dict[str, Any]], force_rebuild: bool = False):
        self.doc_ids = [c["doc_id"] for c in corpus_chunks]
        if not force_rebuild and os.path.exists(DENSE_INDEX_PATH):
            try:
                embs = np.load(DENSE_INDEX_PATH)
                if embs is not None and embs.ndim == 2 and (len(corpus_chunks) == 0 or len(embs) == len(corpus_chunks)):
                    print(f"[Dense] Loaded cached embeddings shape {embs.shape}.")
                    self.corpus_embeddings = embs
                    self.load_model()
                    return
            except Exception:
                pass

        self.load_model()
        if self.model is None or not corpus_chunks:
            return

        texts = [c["text"] for c in corpus_chunks]
        print(f"[Dense] Computing dense embeddings for {len(texts)} chunks...")
        batch_size = 128 if self.device == "cuda" else 64
        embeddings = self.model.encode(texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
        self.corpus_embeddings = np.array(embeddings, dtype=np.float32)
        np.save(DENSE_INDEX_PATH, self.corpus_embeddings)

    def search(self, question: str, top_k: int = FIRST_STAGE_TOP_K) -> List[Tuple[str, float]]:
        if self.corpus_embeddings is None or len(self.corpus_embeddings) == 0 or self.corpus_embeddings.ndim != 2:
            return []
        if self.model is None:
            self.load_model()
        if self.model is None:
            return []

        query_emb = self.model.encode([question], normalize_embeddings=True)[0]
        scores = np.dot(self.corpus_embeddings, query_emb)
        doc_scores: Dict[str, float] = {}
        for idx, score in enumerate(scores):
            if idx < len(self.doc_ids):
                d_id = self.doc_ids[idx]
                if d_id not in doc_scores or score > doc_scores[d_id]:
                    doc_scores[d_id] = float(score)
        return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [ ]:
# =====================================================================
# 7. STEP 4: DEEP RE-RANKING (CROSS-ENCODER WITH 10X SPEEDUP)
# =====================================================================
class LegalReranker:
    def __init__(self, model_name: str = RERANKER_MODEL_NAME):
        self.model_name = model_name
        self.model = None
        self.device = DEVICE

    def load_model(self):
        if self.model is None and HAS_CROSS_ENCODER:
            try:
                print(f"[Reranker] Loading Cross-Encoder model ({self.device}): {self.model_name}")
                self.model = CrossEncoder(self.model_name, max_length=RERANK_MAX_LENGTH, device=self.device)
            except Exception as e:
                print(f"[Reranker] Warning: Could not load model '{self.model_name}': {e}")
                self.model = None

    def rerank(
        self,
        question: str,
        candidate_doc_ids: List[str],
        doc_chunks_map: Dict[str, List[Dict[str, Any]]],
        first_stage_scores: Dict[str, float] = None,
        top_k_rerank: int = RERANK_TOP_K,
        chunks_per_doc: int = RERANK_CHUNKS_PER_DOC
    ) -> List[Tuple[str, float]]:
        if not candidate_doc_ids:
            return []
        self.load_model()

        if self.model is not None:
            pruned_candidates = candidate_doc_ids[:top_k_rerank]
            remaining_candidates = candidate_doc_ids[top_k_rerank:]
            pairs = []
            pair_doc_ids = []

            for doc_id in pruned_candidates:
                chunks = doc_chunks_map.get(doc_id, [])
                if not chunks:
                    continue
                for chunk in chunks[:chunks_per_doc]:
                    pairs.append((question, chunk["text"]))
                    pair_doc_ids.append(doc_id)

            if pairs:
                batch_size = 64 if self.device == "cuda" else 32
                with torch.inference_mode():
                    scores = self.model.predict(pairs, batch_size=batch_size)
                doc_rerank_scores: Dict[str, float] = {}
                for d_id, score in zip(pair_doc_ids, scores):
                    if d_id not in doc_rerank_scores or score > doc_rerank_scores[d_id]:
                        doc_rerank_scores[d_id] = float(score)

                if remaining_candidates and first_stage_scores:
                    min_s = min(doc_rerank_scores.values()) if doc_rerank_scores else 0.0
                    for d_id in remaining_candidates:
                        doc_rerank_scores[d_id] = min_s - 10.0 + first_stage_scores.get(d_id, 0.0)

                return sorted(doc_rerank_scores.items(), key=lambda x: x[1], reverse=True)

        if first_stage_scores is not None:
            return sorted([(d, first_stage_scores.get(d, 0.0)) for d in candidate_doc_ids], key=lambda x: x[1], reverse=True)
        return [(d, 1.0 / (idx + 1)) for idx, d in enumerate(candidate_doc_ids)]

In [ ]:
# =====================================================================
# 8. PIPELINE INFERENCE & DYNAMIC THRESHOLDING
# =====================================================================
def build_doc_chunks_map(corpus_chunks: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    doc_map: Dict[str, List[Dict[str, Any]]] = {}
    for chunk in corpus_chunks:
        d_id = chunk["doc_id"]
        if d_id not in doc_map:
            doc_map[d_id] = []
        doc_map[d_id].append(chunk)
    return doc_map

def run_pipeline_for_question(
    question: str,
    bm25_retriever: BM25Retriever,
    dense_retriever: DenseRetriever,
    reranker: LegalReranker,
    doc_chunks_map: Dict[str, List[Dict[str, Any]]],
    threshold_ratio: float = DYNAMIC_THRESHOLD_RATIO,
    top_k_first_stage: int = FIRST_STAGE_TOP_K
) -> List[str]:
    # 1. Lexical BM25
    bm25_res = bm25_retriever.search(question, top_k=top_k_first_stage)
    bm25_dict = {d: s for d, s in bm25_res}

    # 2. Dense Vector
    dense_res = dense_retriever.search(question, top_k=top_k_first_stage)
    dense_dict = {d: s for d, s in dense_res}

    # 3. Score Fusion
    all_candidates = list(set(list(bm25_dict.keys()) + list(dense_dict.keys())))
    max_bm25 = max(bm25_dict.values()) if bm25_dict and max(bm25_dict.values()) > 0 else 1.0
    max_dense = max(dense_dict.values()) if dense_dict and max(dense_dict.values()) > 0 else 1.0

    combined_scores = {}
    for d in all_candidates:
        norm_b = bm25_dict.get(d, 0.0) / max_bm25
        norm_d = dense_dict.get(d, 0.0) / max_dense
        combined_scores[d] = 0.5 * norm_b + 0.5 * norm_d
    sorted_candidates = [d for d, _ in sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:top_k_first_stage]]

    # 4. Cross-Encoder Reranking
    reranked = reranker.rerank(
        question=question,
        candidate_doc_ids=sorted_candidates,
        doc_chunks_map=doc_chunks_map,
        first_stage_scores=combined_scores
    )
    if not reranked:
        return []

    # 5. Dynamic Thresholding
    top1_doc, top1_score = reranked[0]
    predicted_docs = [top1_doc]
    threshold = top1_score * threshold_ratio
    for d, s in reranked[1:]:
        if s >= threshold and len(predicted_docs) < MAX_PREDICTED_DOCS:
            predicted_docs.append(d)
        else:
            break
    return predicted_docs

In [ ]:
# =====================================================================
# 9. EVALUATION ON VALIDATION SPLIT (1400 SAMPLES)
# =====================================================================
def evaluate_pipeline():
    print("=" * 60)
    print("      EVALUATING LEGAL DOCUMENT RETRIEVAL PIPELINE      ")
    print("=" * 60)
    train_data = load_json(TRAIN_PATH)
    items = list(train_data.items())
    random.seed(RANDOM_SEED)
    random.shuffle(items)
    val_size = int(len(items) * VAL_RATIO)
    val_data = dict(items[:val_size])

    corpus_chunks = process_corpus()
    doc_chunks_map = build_doc_chunks_map(corpus_chunks)

    bm25 = BM25Retriever()
    bm25.build_index(corpus_chunks)

    dense = DenseRetriever()
    dense.build_index(corpus_chunks)

    reranker = LegalReranker()

    print(f"\n[Validation] Evaluating on {len(val_data)} samples with optimized settings...")
    predictions = {}
    for q_id, sample in tqdm(val_data.items(), desc="Evaluating"):
        question = sample.get("question", "")
        pred_answers = run_pipeline_for_question(
            question=question,
            bm25_retriever=bm25,
            dense_retriever=dense,
            reranker=reranker,
            doc_chunks_map=doc_chunks_map
        )
        predictions[q_id] = {"question": question, "answer": pred_answers}

    mean_recall, mean_precision = evaluate_predictions(val_data, predictions)
    print("\n" + "=" * 60)
    print(f"  Primary Metric   - Mean Recall    : {mean_recall:.4f}")
    print(f"  Secondary Metric - Mean Precision : {mean_precision:.4f}")
    print("=" * 60)
    return mean_recall, mean_precision

In [ ]:
# =====================================================================
# 10. GENERATE TEST SUBMISSION & CREATE SUBMISSION.ZIP
# =====================================================================
def generate_submission():
    print("=" * 60)
    print("      GENERATING FINAL SUBMISSION FOR PUBLIC TEST       ")
    print("=" * 60)
    test_data = load_json(PUBLIC_TEST_PATH)
    print(f"[Predict] Loaded {len(test_data)} test questions.")

    corpus_chunks = process_corpus()
    doc_chunks_map = build_doc_chunks_map(corpus_chunks)

    bm25 = BM25Retriever()
    bm25.build_index(corpus_chunks)

    dense = DenseRetriever()
    dense.build_index(corpus_chunks)

    reranker = LegalReranker()

    predictions = {}
    for q_id, sample in tqdm(test_data.items(), desc="Predicting Test Set"):
        question = sample.get("question", "")
        preds = run_pipeline_for_question(
            question=question,
            bm25_retriever=bm25,
            dense_retriever=dense,
            reranker=reranker,
            doc_chunks_map=doc_chunks_map
        )
        if not preds:
            preds = ["100050"]
        predictions[q_id] = {"answer": [str(a) for a in preds[:5]]}

    # Sanity Checks
    assert len(predictions) == len(test_data)
    null_cnt = sum(1 for v in predictions.values() if not v.get("answer"))
    over_cnt = sum(1 for v in predictions.values() if len(v.get("answer", [])) > 5)
    print(f"[Sanity Check] Total: {len(predictions)} | Nulls: {null_cnt} | Over limit: {over_cnt}")

    save_json(predictions, SUBMISSION_PATH, indent=4)
    with zipfile.ZipFile(SUBMISSION_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(SUBMISSION_PATH, arcname="submission.json")
    print(f"[Success] Created {SUBMISSION_ZIP_PATH} ready for submission!")
    return predictions

In [ ]:
# =====================================================================
# 11. EXECUTE PIPELINE
# =====================================================================
# Uncomment the action you wish to run:

# 1. Run Evaluation on Validation Split:
# evaluate_pipeline()

# 2. Run Final Prediction on Public Test set (Generate submission.zip):
generate_submission()